In [ ]:
'''
Machine learning and Agent-to-agent (A2A) protocols

Dataset: Iris Dataset (https://archive.ics.uci.edu/ml/datasets/Iris?utm_source=chatgpt.com)

Agent to Agent Protocols:
A2A allows agents to communicate and delegate tasks using standardized messages (typically
JSON-RPC over HTTP). Agents publish capabilities and exchange tasks rather than sharing
internal logic.

For this project we will create:

Agent 1 – Data Agent
Responsibilities:
- Load dataset
- Clean dataset
- Compute statistics
- Agent 2 – ML Agent

Responsibilities:
- Train machine learning model
- Predict species
- Agent 3 – Reporting Agent

Responsibilities:
- Receive results
- Generate final report

The agents communicate through A2A-style JSON messages.

Project: The code below is a simplified A2A architecture using Python.
It follows the concepts of Agent Cards, task delegation, and agent-to-agent
messaging used in A2A systems.

This project demonstrates several core A2A concepts:
- Agent Cards:  Each agent advertises its capabilities.
- Task Delegation: DataAgent delegates model training to MLAgent.
- Agent Messaging: Structured JSON messages are exchanged.
- Multi-Agent Workflow: DataAgent → MLAgent → ReportingAgent.
- Machine Learning: Random Forest classifier trained on the Iris dataset.
- Google Colab Compatible: Runs in a single notebook without additional infrastructure.
'''


In [15]:
# INSTALL DEPENDENCIES

!pip -q install pandas scikit-learn

print ("dependencies ready")

dependencies ready


In [16]:
# IMPORT LIBRARIES

import json
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print ("Libraries are ready.")

Libraries are ready.


In [17]:
# DEFINE A2A MESSAGE STRUCTURE

class A2AMessage:

    def __init__(self, sender, receiver, task, payload):
        self.sender = sender
        self.receiver = receiver
        self.task = task
        self.payload = payload

    def to_json(self):
        return json.dumps({
            "sender": self.sender,
            "receiver": self.receiver,
            "task": self.task,
            "payload": self.payload
        }, indent=2)

print("A2A message structure ready")

A2A message structure ready


In [18]:
# DEFINE AGENT CARD

class AgentCard:

    def __init__(self, name, description, capabilities):
        self.name = name
        self.description = description
        self.capabilities = capabilities

    def display(self):
        print(f"\nAgent: {self.name}")
        print(f"Description: {self.description}")
        print(f"Capabilities: {self.capabilities}")

print ("Agent card defined")

Agent card defined


In [19]:
# DATA AGENT

class DataAgent:

    def __init__(self):

        self.card = AgentCard(
            "DataAgent",
            "Loads and prepares datasets",
            ["load_data", "prepare_data"]
        )

    def process(self):

        iris = load_iris(as_frame=True)

        df = iris.frame

        print("\n[DataAgent] Dataset Loaded")
        print(df.head())

        message = A2AMessage(
            sender="DataAgent",
            receiver="MLAgent",
            task="train_model",
            payload=df.to_dict()
        )

        return message

print("Data agent ready")

Data agent ready


In [20]:
# MACHINE LEARNING AGENT

class MLAgent:

    def __init__(self):

        self.card = AgentCard(
            "MLAgent",
            "Trains classification models",
            ["train", "predict"]
        )

    def process(self, message):

        print("\n[MLAgent] Message Received")
        print(message.to_json())

        df = pd.DataFrame(message.payload)

        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )

        model = RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)

        print(f"\n[MLAgent] Accuracy: {accuracy:.4f}")

        report_message = A2AMessage(
            sender="MLAgent",
            receiver="ReportingAgent",
            task="generate_report",
            payload={
                "accuracy": float(accuracy),
                "rows": len(df),
                "features": len(X.columns)
            }
        )

        return report_message
print("ML agent ready")

ML agent ready


In [22]:
# REPORTING AGENT

class ReportingAgent:

    def __init__(self):

        self.card = AgentCard(
            "ReportingAgent",
            "Creates business reports",
            ["reporting"]
        )

    def process(self, message):

        print("\n[ReportingAgent] Message Received")
        print(message.to_json())

        report = f"""
================================================
MULTI-AGENT MACHINE LEARNING REPORT
================================================

Dataset Rows: {message.payload['rows']}
Features: {message.payload['features']}
Model Accuracy: {message.payload['accuracy']:.4f}

Status:
SUCCESS

Agents Involved:
1. DataAgent
2. MLAgent
3. ReportingAgent

Workflow:
DataAgent -> MLAgent -> ReportingAgent

================================================
"""

        print(report)

In [23]:
# AGENT DISCOVERY

data_agent = DataAgent()
ml_agent = MLAgent()
reporting_agent = ReportingAgent()

print("AGENT CARDS")

data_agent.card.display()
ml_agent.card.display()
reporting_agent.card.display()


AGENT CARDS

Agent: DataAgent
Description: Loads and prepares datasets
Capabilities: ['load_data', 'prepare_data']

Agent: MLAgent
Description: Trains classification models
Capabilities: ['train', 'predict']

Agent: ReportingAgent
Description: Creates business reports
Capabilities: ['reporting']


In [24]:
# EXECUTE A2A WORKFLOW

print("\n\nSTARTING A2A WORKFLOW")

msg1 = data_agent.process()

msg2 = ml_agent.process(msg1)

reporting_agent.process(msg2)




STARTING A2A WORKFLOW

[DataAgent] Dataset Loaded
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

[MLAgent] Message Received
{
  "sender": "DataAgent",
  "receiver": "MLAgent",
  "task": "train_model",
  "payload": {
    "sepal length (cm)": {
      "0": 5.1,
      "1": 4.9,
      "2": 4.7,
      "3": 4.6,
      "4": 5.0,
      "5": 5.4,
      "6": 4.6,
      "7": 5.0,
      "8": 4.4,
      "9": 4.9,
      "10": 5.4,
      "11": 4.8,
      "12": 4.8,
      "13": 4.3,
      "14": 5.8,
      "15

In [25]:
# END

print("\nA2A WORKFLOW COMPLETED")



A2A WORKFLOW COMPLETED
